In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 40.3 MB/s eta 0:00:00


In [20]:
import torch
from torch import nn, optim
import re
from torch.utils.data import TensorDataset, DataLoader, random_split
import gensim.downloader as api
from datasets import load_dataset

In [3]:
glove = api.load('glove-wiki-gigaword-50')

[==================================================] 100.0% 66.0/66.0MB downloaded


In [4]:
# Use GPU instead of CPU for faster results
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [14]:
dataset = load_dataset('IMDB')
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [6]:
def tokenize(text):
  text = re.sub(r'<.*?>', ' ', text)  # remove HTML tags
  text = text.lower()
  text = text.replace("'", " '").replace("!", " !").replace("?", " ?")
  return text.split()

In [7]:
class Vocabulary():
  def __init__(self):
    self.wordmap = {
        '<pad>' : 0,
        '<unk>' : 1
    }

  def build(self, sentences):
    for sentence in sentences:
      sentence = tokenize(sentence)
      for word in sentence:
        if word not in self.wordmap.keys():
          self.wordmap[word] = max(self.wordmap.values()) + 1

  def to_indices(self, text):
    indices = []
    for word in tokenize(text):
      if word not in self.wordmap.keys():
        indices.append(1)
      else:
        indices.append(self.wordmap[word])
    return indices

In [15]:
def extract_common_words(limit=10000):
  reviews = dataset['train']['text']
  word_counts = dict()

  for review in reviews:
    review = tokenize(review)
    for word in review:
      if word not in word_counts.keys():
        word_counts[word] = 1
      else:
        word_counts[word] += 1

  sorted_words = sorted(word_counts, key=word_counts.get, reverse=True)

  return sorted_words[:limit]

In [10]:
words = extract_common_words()

In [11]:
vocab = Vocabulary()
vocab.build(words)

In [22]:
class BidirectionalLSTMSentimentModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding = nn.Embedding(num_embeddings=len(vocab.wordmap), embedding_dim=50)
    self.bilstm = nn.LSTM(input_size=50, hidden_size=128, dropout=0.3, num_layers=2, batch_first=True, bidirectional=True)
    self.dropout = nn.Dropout(p=0.3)
    self.linear = nn.Linear(256, 1)

  def forward(self, x):
    x = self.embedding(x)
    out, (hidden, cell) = self.bilstm(x)
    hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
    hidden = self.dropout(hidden)
    x = torch.sigmoid(self.linear(hidden))
    return x

In [12]:
def get_review_indices(review):
  indices = vocab.to_indices(review)
  if len(indices) > 256:
    indices = indices[:256]
  else:
    indices += [0] * (256 - len(indices))  # 0 = <pad> index
  return indices

In [27]:
# Prepare data for training/testing
review_tensors_train = []
labels_train = dataset['train']['label']

review_tensors_test = []
labels_test = dataset['test']['label']

for review in dataset['train']['text']:
  review_tensors_train.append(torch.tensor(get_review_indices(review), dtype=torch.long))

for review in dataset['test']['text']:
  review_tensors_test.append(torch.tensor(get_review_indices(review), dtype=torch.long))

X_train = torch.stack(review_tensors_train)
y_train = torch.tensor(labels_train, dtype=torch.float32).reshape(-1, 1)

X_test = torch.stack(review_tensors_test)
y_test = torch.tensor(labels_test, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train, y_train)
train_set, val_set = random_split(train_dataset, [20000, 5000])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False)

test_set = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

In [31]:
model = BidirectionalLSTMSentimentModel().to(device)

In [32]:
loss_ft = nn.BCELoss()
opt = optim.Adam(model.parameters(), lr=0.01)

In [33]:
for epoch in range(3):
  train_batch_loss = 0
  val_batch_loss = 0

  model.train()
  for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    pred_train = model(X_batch)
    loss_train = loss_ft(pred_train, y_batch)
    loss_train.backward()
    opt.step()
    opt.zero_grad()

    train_batch_loss += loss_train.item()

  avg_train_batch_loss = train_batch_loss / len(train_loader)


  model.eval()
  with torch.no_grad():
    for X_batch, y_batch in val_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      pred_val = model(X_batch)
      loss_val = loss_ft(pred_val, y_batch)

      val_batch_loss += loss_val.item()

    avg_val_batch_loss = val_batch_loss / len(val_loader)


  print(f'Epoch {epoch}')
  print(f'Training set loss: {avg_train_batch_loss:.4f}')
  print(f'Validation set loss: {avg_val_batch_loss:.4f}')
  print(f'Difference: {(avg_train_batch_loss - avg_val_batch_loss):.4f}')
  print('\n')

Epoch 0
Training set loss: 0.6931
Validation set loss: 0.7106
Difference: -0.0176


Epoch 1
Training set loss: 0.6223
Validation set loss: 0.5318
Difference: 0.0905


Epoch 2
Training set loss: 0.4066
Validation set loss: 0.3560
Difference: 0.0505




In [35]:
total_correct = 0

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        preds = (model(X_batch) > 0.5).float()
        total_correct += (preds == y_batch).sum().item()

acc = total_correct / len(test_set)
print(f"Test accuracy: {(acc * 100):.4f}%")

Test accuracy: 83.0160%
